In [1]:
import mysql.connector
from credenciales import mysql_config
import pandas as pd

#Carga CSVs para poder crear las tablas SQL
esyrce  = pd.read_csv("DF_procesados/ESYRCE_procesado.csv")
nasa    = pd.read_csv("DF_procesados/NASA_procesado.csv")
aemet   = pd.read_csv("DF_procesados/AEMET_procesado.csv")
faostat = pd.read_csv("DF_procesados/FAOSTAT_procesado.csv")

#### Creación de tablas en SQL
```sql
Tabla ESYRCE 
CREATE TABLE ESYRCE (
    Categoria   VARCHAR(100),
    Cultivo     VARCHAR(100),
    Provincia   VARCHAR(100),
    Año        INT,
    Hectareas   FLOAT,
    PRIMARY KEY (Cultivo, Provincia, Anio)
);

Tabla AEMET
CREATE TABLE AEMET (
    Anio        INT,
    Temp_media  FLOAT,
    Temp_min    FLOAT,
    Temp_max    FLOAT,
    Precipitacion   FLOAT,
    Provincia   VARCHAR(100),
    PRIMARY KEY (Provincia, Anio)
);

Tabla NASA
CREATE TABLE NASA (
    YEAR        INT,
    PRECTOTCORR FLOAT,
    RH2M        FLOAT,
    T2M         FLOAT,
    T2M_MAX     FLOAT,
    T2M_MIN     FLOAT,
    PRIMARY KEY (YEAR)
);

Tabla FAOSTAT
CREATE TABLE FAOSTAT(
    Area        VARCHAR(100),
    Elemento    VARCHAR(100),
    Producto    VARCHAR(100),
    Anio        INT,
    Unidad      VARCHAR(100),
    Valor       FLOAT,
    Simbolo     VARCHAR(100),
    Descripcion_Simbolo     VARCHAR(100),
    PRIMARY KEY (Producto, Elemento, Anio)
);
```

In [27]:
#Función para crear las tablas en TiDB.
def cargar_csv_en_tabla(df, tabla, conn, batch=1000):
    """Inserta un DataFrame en una tabla de TiDB en lotes de `batch` filas."""
    cursor = conn.cursor()
    cols = ", ".join(df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))
    sql = f"INSERT INTO {tabla} ({cols}) VALUES ({placeholders})"

    filas = [tuple(None if pd.isna(v) else v for v in row)
             for row in df.itertuples(index=False, name=None)]

    for i in range(0, len(filas), batch):
        cursor.executemany(sql, filas[i:i+batch])
        conn.commit()
        print(f"  insertadas {min(i+batch, len(filas))}/{len(filas)}")
    cursor.close()
conn = mysql.connector.connect(**mysql_config)

In [3]:
#Cambio la columna año por anio
esyrce = esyrce.reset_index(drop=True)
esyrce = esyrce.rename(columns={"Año": "Anio"})

In [4]:
#Elimino el índice por problemas en la carga de la tabla
esyrce = esyrce.drop(columns=["Unnamed: 0"])

In [5]:
print(esyrce.columns.tolist())

['Categoria', 'Cultivo', 'Provincia', 'Anio', 'Hectareas']


In [8]:
duplicados = esyrce[esyrce.duplicated(subset=["Categoria", "Cultivo", "Provincia", "Anio"], keep=False)]
print(len(duplicados))
duplicados.sort_values(["Cultivo", "Provincia", "Anio"]).head(20)

882


,Categoria,Cultivo,Provincia,Anio,Hectareas
1860,Hortalizas y flores,MAIZ DULCE,ALBACETE,2004,NaN
1885,Hortalizas y flores,MAIZ DULCE,ALBACETE,2004,NaN
7172,Hortalizas y flores,MAIZ DULCE,ALBACETE,2005,1637.6368
7197,Hortalizas y flores,MAIZ DULCE,ALBACETE,2005,NaN
12484,Hortalizas y flores,MAIZ DULCE,ALBACETE,2006,511.0582
12509,Hortalizas y flores,MAIZ DULCE,ALBACETE,2006,NaN
17796,Hortalizas y flores,MAIZ DULCE,ALBACETE,2007,503.6505
17821,Hortalizas y flores,MAIZ DULCE,ALBACETE,2007,NaN
23108,Hortalizas y flores,MAIZ DULCE,ALBACETE,2008,721.2008
23133,Hortalizas y flores,MAIZ DULCE,ALBACETE,2008,NaN


In [9]:
#Limpieza de duplicados por problemas en la carga de la tabla en SQL
esyrce = esyrce.sort_values("Hectareas", ascending=False)
esyrce = esyrce.drop_duplicates(subset=["Categoria", "Cultivo", "Provincia", "Anio"], keep="first")
esyrce = esyrce.sort_index()

print(len(esyrce))

111111


In [10]:
duplicados = esyrce[esyrce.duplicated(subset=["Categoria", "Cultivo", "Provincia", "Anio"], keep=False)]
print(len(duplicados))

0


In [15]:
cargar_csv_en_tabla(esyrce,'ESYRCE', conn)

  insertadas 1000/111111
  insertadas 2000/111111
  insertadas 3000/111111
  insertadas 4000/111111
  insertadas 5000/111111
  insertadas 6000/111111
  insertadas 7000/111111
  insertadas 8000/111111
  insertadas 9000/111111
  insertadas 10000/111111
  insertadas 11000/111111
  insertadas 12000/111111
  insertadas 13000/111111
  insertadas 14000/111111
  insertadas 15000/111111
  insertadas 16000/111111
  insertadas 17000/111111
  insertadas 18000/111111
  insertadas 19000/111111
  insertadas 20000/111111
  insertadas 21000/111111
  insertadas 22000/111111
  insertadas 23000/111111
  insertadas 24000/111111
  insertadas 25000/111111
  insertadas 26000/111111
  insertadas 27000/111111
  insertadas 28000/111111
  insertadas 29000/111111
  insertadas 30000/111111
  insertadas 31000/111111
  insertadas 32000/111111
  insertadas 33000/111111
  insertadas 34000/111111
  insertadas 35000/111111
  insertadas 36000/111111
  insertadas 37000/111111
  insertadas 38000/111111
  insertadas 39000/11

In [21]:
#Elimino índice por problema con la carga de la tabla en SQL
aemet = aemet.drop(columns=["Clima_AEMET_ID"])

In [22]:
cargar_csv_en_tabla(aemet, 'AEMET', conn)

  insertadas 84/84


In [24]:
print(nasa.columns.tolist())

['Unnamed: 0', 'YEAR', 'PRECTOTCORR', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN']


In [25]:
#Elimino el índice por problemas en la carga de la tabla
nasa = nasa.drop(columns=["Unnamed: 0"])

In [28]:
cargar_csv_en_tabla(nasa, 'NASA', conn)

  insertadas 20/20


In [33]:
#Renombro columnas para quitar acentos por problemas en la carga de la tabla en SQL
faostat = faostat.rename(columns={
    "Área": "Area",
    "Año": "Anio",
    "Símbolo": "Simbolo",
    "Descripción del Símbolo": "Descripcion_Simbolo"
})

In [35]:
#Elimino el indice por problemas en la carga de la tabla
faostat = faostat.drop(columns=["Agric_ONU_ID"])

In [37]:
cargar_csv_en_tabla(faostat, 'FAOSTAT', conn)

  insertadas 1000/5904
  insertadas 2000/5904
  insertadas 3000/5904
  insertadas 4000/5904
  insertadas 5000/5904
  insertadas 5904/5904
